# 05 · Gold: esquema estrela e agregado mensal

Lê `mvp_reclamacoes.silver.reclamacoes` e grava no schema `mvp_reclamacoes.gold`:

| Tabela | Conteúdo |
|---|---|
| `fato_reclamacao` | Uma linha por reclamação finalizada contra empresas do recorte bancário |
| `dim_tempo` | Calendário diário de 01/01/2021 a 31/08/2026 |
| `dim_assunto` | Área → Assunto (o produto ou serviço reclamado) |
| `dim_problema` | Grupo Problema → Problema (o tipo de problema) |
| `dim_empresa` | Empresa (nome padronizado) e segmento |
| `dim_local` | UF e região do consumidor |
| `dim_perfil` | Sexo, faixa etária e canal de contratação (junk dimension) |
| `agg_reclamacoes_mensais` | Total de reclamações por mês na plataforma inteira e no recorte (para a participação da P1) |

- **Recorte bancário:** segmentos "Bancos, Financeiras e Administradoras de Cartão" e "Empresas de Pagamento Eletrônico". É regra de negócio, por isso é aplicado aqui e não na Silver.
- **Chaves:** `xxhash64` da chave natural convertida para texto, porque o hash depende do tipo de dado ([xxhash64](https://docs.databricks.com/aws/en/sql/language-manual/functions/xxhash64)). A `dim_tempo` usa o inteiro `aaaammdd`.
- **Carga:** todas as tabelas são recriadas a cada execução.

## 1. Parâmetros

In [ ]:
from pyspark.sql import functions as F

SILVER = "mvp_reclamacoes.silver.reclamacoes"
GOLD = "mvp_reclamacoes.gold"
SEGMENTOS_RECORTE = [
    "Bancos, Financeiras e Administradoras de Cartão",
    "Empresas de Pagamento Eletrônico",
]
INICIO, FIM = "2021-01-01", "2026-08-31"  # período do MVP (dim_tempo)

silver = spark.table(SILVER)
recorte = silver.filter(F.col("segmento_mercado").isin(SEGMENTOS_RECORTE))


def sk(*colunas):
    """Chave substituta: xxhash64 das colunas convertidas para texto."""
    return F.xxhash64(*[F.col(c).cast("string") for c in colunas])


# F = Feminino e M = Masculino pelo dicionário de dados da fonte; O = Outro é definição do projeto
# (o dicionário v1.1 documenta só F e M); nulo vira o membro "Não informado".
sexo_rotulo = (
    F.when(F.col("sexo") == "F", "Feminino")
    .when(F.col("sexo") == "M", "Masculino")
    .when(F.col("sexo") == "O", "Outro")
    .when(F.col("sexo").isNull(), "Não informado")
)
recorte = recorte.withColumn("sexo_rotulo", sexo_rotulo)


def gravar(df, nome):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{nome}")

## 2. Dimensões

In [ ]:
dim_tempo = (
    spark.sql(f"SELECT explode(sequence(DATE'{INICIO}', DATE'{FIM}', INTERVAL 1 DAY)) AS data")
    .select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("sk_tempo"),
        "data",
        F.year("data").alias("ano"),
        F.month("data").alias("mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
    )
)
dim_assunto = recorte.select("area", "assunto").distinct().select(sk("assunto").alias("sk_assunto"), "area", "assunto")
dim_problema = recorte.select("grupo_problema", "problema").distinct().select(
    sk("problema").alias("sk_problema"), "grupo_problema", "problema")
dim_empresa = recorte.select("nome_empresa", "segmento_mercado").distinct().select(
    sk("nome_empresa").alias("sk_empresa"), "nome_empresa", "segmento_mercado")
dim_local = recorte.select("uf", "regiao").distinct().select(sk("uf").alias("sk_local"), "uf", "regiao")
dim_perfil = (
    recorte.select(F.col("sexo_rotulo").alias("sexo"), "faixa_etaria", "canal_contratacao").distinct()
    .select(sk("sexo", "faixa_etaria", "canal_contratacao").alias("sk_perfil"), "sexo", "faixa_etaria", "canal_contratacao")
)

DIMENSOES = {
    # nome: (DataFrame, chave substituta, chave natural)
    "dim_tempo": (dim_tempo, "sk_tempo", ["data"]),
    "dim_assunto": (dim_assunto, "sk_assunto", ["assunto"]),
    "dim_problema": (dim_problema, "sk_problema", ["problema"]),
    "dim_empresa": (dim_empresa, "sk_empresa", ["nome_empresa"]),
    "dim_local": (dim_local, "sk_local", ["uf"]),
    "dim_perfil": (dim_perfil, "sk_perfil", ["sexo", "faixa_etaria", "canal_contratacao"]),
}
for nome, (df, _, _) in DIMENSOES.items():
    gravar(df, nome)
    print(f"gravada {GOLD}.{nome}")

## 3. Fato e agregado mensal

In [ ]:
fato_reclamacao = recorte.select(
    F.date_format("data_finalizacao", "yyyyMMdd").cast("int").alias("sk_tempo"),
    sk("assunto").alias("sk_assunto"),
    sk("problema").alias("sk_problema"),
    sk("nome_empresa").alias("sk_empresa"),
    sk("uf").alias("sk_local"),
    F.xxhash64(F.col("sexo_rotulo").cast("string"), F.col("faixa_etaria").cast("string"),
               F.col("canal_contratacao").cast("string")).alias("sk_perfil"),
    "tempo_resposta_dias",
    "nota_consumidor",
    F.col("respondida").alias("foi_respondida"),
    F.col("nota_consumidor").isNotNull().alias("foi_avaliada"),
    F.when(F.col("avaliacao_reclamacao") == "Resolvida", True)
     .when(F.col("avaliacao_reclamacao") == "Não Resolvida", False).alias("foi_resolvida"),
    "linha_repetida",
)
gravar(fato_reclamacao, "fato_reclamacao")
print(f"gravada {GOLD}.fato_reclamacao")

agg_reclamacoes_mensais = (
    silver.groupBy(F.date_format("data_finalizacao", "yyyy-MM").alias("ano_mes"))
    .agg(
        F.count("*").alias("total_plataforma"),
        F.sum(F.col("segmento_mercado").isin(SEGMENTOS_RECORTE).cast("int")).alias("total_recorte"),
    )
    .orderBy("ano_mes")
)
gravar(agg_reclamacoes_mensais, "agg_reclamacoes_mensais")
print(f"gravada {GOLD}.agg_reclamacoes_mensais")

## 4. Verificações

1. **Sem colisão de hash:** em cada dimensão, o número de chaves substitutas distintas é igual ao número de linhas.
2. **Chave natural única:** em cada dimensão, o número de chaves naturais distintas é igual ao número de linhas. Isso reconfirma na Silver as hierarquias da H1 e da H5.
3. **Integridade referencial:** toda chave da fato existe na dimensão correspondente.
4. **Linhas da fato** = linhas da Silver no recorte.
5. **Agregado mensal:** 68 meses; soma de `total_recorte` = linhas da fato; soma de `total_plataforma` = linhas da Silver.

As contagens esperadas vêm do perfilamento (notebook 03) e são só impressas para comparação.

In [ ]:
ESPERADO = {"dim_tempo": 2069, "dim_assunto": 116, "dim_problema": 144, "dim_empresa": 457, "dim_local": 27, "dim_perfil": 208}
fato = spark.table(f"{GOLD}.fato_reclamacao")

for nome, (_, chave, naturais) in DIMENSOES.items():
    dim = spark.table(f"{GOLD}.{nome}")
    linhas = dim.count()
    chaves = dim.select(chave).distinct().count()
    naturais_distintas = dim.select(*naturais).distinct().count()
    orfas = fato.join(dim, chave, "left_anti").count()
    print(f"{nome:<14} linhas={linhas:>6,} (esperado {ESPERADO[nome]:>5,}) | sk distintas={chaves:>6,} | "
          f"chave natural distinta={naturais_distintas:>6,} | chaves da fato sem dimensão={orfas:,}")
    assert chaves == linhas, f"{nome}: colisão de hash na chave substituta"
    assert naturais_distintas == linhas, f"{nome}: chave natural repetida (hierarquia quebrada)"
    assert orfas == 0, f"{nome}: a fato tem chaves que não existem na dimensão"

linhas_fato, linhas_recorte = fato.count(), recorte.count()
agg = spark.table(f"{GOLD}.agg_reclamacoes_mensais")
resumo = agg.agg(F.count("*").alias("meses"), F.sum("total_recorte").alias("recorte"), F.sum("total_plataforma").alias("plataforma")).first()
linhas_silver = silver.count()
print(f"\nfato_reclamacao: {linhas_fato:,} linhas | Silver no recorte: {linhas_recorte:,}")
print(f"agg_reclamacoes_mensais: {resumo['meses']} meses | soma total_recorte={resumo['recorte']:,} | "
      f"soma total_plataforma={resumo['plataforma']:,} | linhas da Silver={linhas_silver:,}")
assert linhas_fato == linhas_recorte, "a fato precisa ter todas as linhas do recorte"
assert resumo["meses"] == 68, "esperados 68 meses no agregado"
assert resumo["recorte"] == linhas_fato, "soma de total_recorte difere das linhas da fato"
assert resumo["plataforma"] == linhas_silver, "soma de total_plataforma difere das linhas da Silver"
print("\nTodas as verificações passaram.")